# Importing Libraries

In [ ]:
import os
import numpy as np
import nibabel as nib
import torch
from tqdm import tqdm

# Dataset (sagittal)

In [ ]:
directory_path = "DS_final/sagittal"
output_path = "dataset.pt"
num_images = 10000

We will tensorise our sagittal dataset by turning the NIfTI slices into numpy arrays and then into a single tensor.

I can't use the `ImageFolder()` class from torch because the slices are NIfTI files, not images with a common extension such as JPEG, so the loading is done from scratch.

In [ ]:
# Sorted so the same 10,000 slices are picked on every run
files = sorted(f for f in os.listdir(directory_path) if f.endswith(".nii.gz"))
print(len(files), "slices found")

# Normalisation

Every slice has its own maximum intensity, so a single global normalisation doesn't work. Each slice is scaled to [-1, 1] using its own maximum, which matches the `tanh` output of the generator.

In [ ]:
def normalize(image_arr):
    mx = image_arr.max()
    if mx == 0:
        return np.full_like(image_arr, -1.0)
    return image_arr / (mx / 2) - 1

# Transforming all arrays into one tensor

I only take the first 10,000 slices as the dataset because this step is memory heavy.

In [ ]:
def tensorize(directory_path, files):
    data = np.empty((len(files), 256, 256), dtype=np.float32)
    for i, filename in enumerate(tqdm(files)):
        img_arr = nib.load(os.path.join(directory_path, filename)).get_fdata(dtype=np.float32)
        data[i] = normalize(img_arr.reshape(256, 256))
    return torch.from_numpy(data)

In [ ]:
dataset = tensorize(directory_path, files[:num_images])

In [ ]:
torch.save(dataset, output_path)

In [ ]:
dataset.shape, dataset.min(), dataset.max()